# HalluGuard: Real-Time Hallucination Detection & Mitigation
## Evaluation Results — First Project Review

In [ ]:
import json
import os
import sys

# Ensure project root is on path
_NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
if 'experiments' not in _NOTEBOOK_DIR:
    _NOTEBOOK_DIR = os.path.join(_NOTEBOOK_DIR, 'experiments')
_PROJECT_ROOT = os.path.dirname(_NOTEBOOK_DIR)
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)

RESULTS_JSON = os.path.join(_NOTEBOOK_DIR, 'results', 'quick_demo_results.json')

with open(RESULTS_JSON, 'r') as f:
    data = json.load(f)

metrics = data['metrics']
bl = metrics['baseline']
hg = metrics['halluguard']
tiers = metrics['mitigation_tier_counts']
n = metrics['num_samples']
best = data.get('best_improvements', [])

# ── Main Metrics Table ──
try:
    from tabulate import tabulate
    headers = ['Metric', 'Baseline RAG', 'HalluGuard']
    rows = [
        ['Hallucination Rate', f"{bl['hallucination_rate_pct']:.2f}%", f"{hg['hallucination_rate_pct']:.2f}%"],
        ['Mean Faithfulness', f"{bl['mean_faithfulness']:.4f}", f"{hg['mean_faithfulness']:.4f}"],
        ['AUROC', f"{bl['auroc']:.4f}", f"{hg['auroc']:.4f}"],
        ['Mean Latency (ms)', f"{bl['latency_mean_ms']:.2f}", f"{hg['latency_mean_ms']:.2f}"],
        ['p95 Latency (ms)', f"{bl['latency_p95_ms']:.2f}", f"{hg['latency_p95_ms']:.2f}"],
        ['', '', ''],
        ['Halluc. Reduction', '—', f"-{bl['hallucination_rate_pct'] - hg['hallucination_rate_pct']:.1f} pts"],
        ['Faith. Improvement', '—', f"+{hg['mean_faithfulness'] - bl['mean_faithfulness']:.4f}"],
    ]
    print(tabulate(rows, headers=headers, tablefmt='pretty'))
except ImportError:
    print('Install tabulate: pip install tabulate')

# ── Mitigation Tier Breakdown ──
print('\nMitigation Tier Breakdown:')
for tier, count in sorted(tiers.items()):
    print(f'  {tier:20s} {count:3d}  ({count/n*100:.1f}%)')

In [ ]:
from IPython.display import Image, display
import os

img_path = os.path.join('experiments', 'results', 'halluguard_results.png')
if not os.path.exists(img_path):
    img_path = os.path.join('results', 'halluguard_results.png')

display(Image(filename=img_path, width=900))

## Top 3 Improvement Cases

The following cases show the largest hallucination-score reduction
between the baseline RAG response and the HalluGuard-mitigated response.

In [ ]:
from IPython.display import Markdown, display

for rank, case in enumerate(best[:3], start=1):
    improvement = case.get('improvement', 0)
    bl_resp = case.get('baseline_response', 'N/A')[:300]
    hg_resp = case.get('halluguard_response', 'N/A')[:300]
    md = f"""
### Case #{rank} — Improvement: {improvement:+.4f}

| | |
|---|---|
| **Question** | {case.get('question', 'N/A')} |
| **Baseline Score** | {case.get('baseline_score', 'N/A'):.4f} |
| **HalluGuard Score** | {case.get('halluguard_score', 'N/A'):.4f} |
| **Strategy** | {case.get('strategy_used', 'N/A')} |

**Baseline Response:** {bl_resp}{'…' if len(case.get('baseline_response', '')) > 300 else ''}

**HalluGuard Response:** {hg_resp}{'…' if len(case.get('halluguard_response', '')) > 300 else ''}

---
"""
    display(Markdown(md))

In [ ]:
print(f"""\n{'='*60}""")
print("CONCLUSION")
print(f"{'='*60}\n")

bl_rate = bl['hallucination_rate_pct']
hg_rate = hg['hallucination_rate_pct']
relative_reduction = (bl_rate - hg_rate) / bl_rate * 100 if bl_rate > 0 else 0

print(
    f"HalluGuard reduced hallucination rate from {bl_rate:.0f}% to {hg_rate:.0f}% "
    f"({relative_reduction:.1f}% relative reduction) with mean faithfulness improving "
    f"from {bl['mean_faithfulness']:.2f} to {hg['mean_faithfulness']:.2f} "
    f"on {n} HaluEval QA samples."
)
print(f"\nMean per-sample latency: {hg['latency_mean_ms']:.0f} ms (HalluGuard) vs {bl['latency_mean_ms']:.0f} ms (Baseline).")
print("\n✅ All evaluation complete.")